# **Custom YOLOv8 Training for Pothole Detection (with ONNX Export for Hailo)**




**Verify GPU Availability**

Make sure you're using a GPU-equipped machine by going to "Runtime" -> "Change runtime type" in the top menu bar, and then selecting one of the GPU options in the Hardware accelerator section. Click Play on the following code block to verify that the NVIDIA GPU is present and ready for training.

In [ ]:
!nvidia-smi

### **1. Install Dependencies**

In [ ]:
# 1. Install necessary libraries
!uv pip install ultralytics
import ultralytics
ultralytics.checks()


### **2. Download Dataset from Roboflow (YOLOv8 format)**

Replace *YOUR_API_KEY, YOUR_WORKSPACE, YOUR_PROJECT*, and *VERSION_NUMBER* with your actual Roboflow details.

If you haven't already, you can find your API key in your Roboflow account settings.

In [ ]:
!pip install roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="your_API_Key")
project = rf.workspace("your_workspace").project("your_project")
version = project.version(version_number)
dataset = version.download("yolov8")



# Set the path to the dataset's data.yaml file
#data_yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f"Dataset downloaded to: {dataset.location}")

print("Dataset download complete. Next, we will train the YOLOv8 model.")

This will create a folder (e.g., `Pothole-Detection-1`) containing `train/`, `valid/`, `test/` folders and a `data.yaml` file with correct class names.

 ### **3. Verify Dataset Structure**

In [ ]:
import os
print(f"Dataset location:", dataset.location)
print("Dataset version:", dataset.version)
!ls {dataset.location}

print("Dataset download complete. Next, we will train the YOLOv8 model.")

You should see `train/`, `valid/`, `data.yaml`.

In [ ]:
import os
import shutil
from pathlib import Path

dataset_path = Path(dataset.location)

# Original folders
train_labels = dataset_path / 'train' / 'labels'
valid_labels = dataset_path / 'valid' / 'labels'
test_labels  = dataset_path / 'test' / 'labels'

# Filtered output
filtered_path = dataset_path / 'filtered'
filtered_train = filtered_path / 'train'
filtered_valid = filtered_path / 'valid'
filtered_test  = filtered_path / 'test'
filtered_train.mkdir(parents=True, exist_ok=True)
filtered_valid.mkdir(parents=True, exist_ok=True)
filtered_test.mkdir(parents=True, exist_ok=True)

def filter_segmentation_files(label_dir, image_dir, out_label_dir, out_image_dir):
    for label_file in label_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            lines = f.readlines()
        is_seg = any(len(line.strip().split()) > 5 for line in lines)
        if is_seg:
            shutil.copy(label_file, out_label_dir / label_file.name)
            img_file = image_dir / (label_file.stem + '.jpg')
            if img_file.exists():
                shutil.copy(img_file, out_image_dir / img_file.name)

# --- Filter Train ---
os.makedirs(filtered_train / 'labels', exist_ok=True)
os.makedirs(filtered_train / 'images', exist_ok=True)
filter_segmentation_files(train_labels, train_images,
                          filtered_train / 'labels', filtered_train / 'images')

# --- Filter Valid ---
os.makedirs(filtered_valid / 'labels', exist_ok=True)
os.makedirs(filtered_valid / 'images', exist_ok=True)
filter_segmentation_files(valid_labels, valid_images,
                          filtered_valid / 'labels', filtered_valid / 'images')

# --- Filter Test ---
os.makedirs(filtered_test / 'labels', exist_ok=True)
os.makedirs(filtered_test / 'images', exist_ok=True)
filter_segmentation_files(test_labels, test_images,
                          filtered_test / 'labels', filtered_test / 'images')

n_train = len(list((filtered_train / 'images').glob('*.jpg')))
n_val   = len(list((filtered_valid / 'images').glob('*.jpg')))
n_test  = len(list((filtered_test / 'images').glob('*.jpg')))
print(f"Filtered images -> Train: {n_train}, Val: {n_val}, Test: {n_test}")

# Write data.yaml (train + val only; test is kept separate)
data_yaml_content = f"""\
path: {filtered_path}
train: train/images
val: valid/images
nc: 1
names: ['pothole']
"""
with open(filtered_path / 'data.yaml', 'w') as f:
    f.write(data_yaml_content)

print(f"Filtered dataset ready at: {filtered_path}")
print(f"New data.yaml: {filtered_path}/data.yaml")


#Create a calibration folder and copy a subset of filtered images (from train)
calib_path = dataset_path/'calibration_set'
calib_path.mkdir(parents=True,exist_ok=True)

#Copy 1000 images from filtered_train/images
import random
train_images_filtered = list((filtered_train / 'images').glob('*.jpg'))
sample = random.sample(train_images_filtered,min(1000,len(train_images_filtered)))
for img_path in sample:
  shutil.copy(img_path, calib_path/img_path.name)

print(f"Calibration images copied to {calib_path}")

from google.colab import drive
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/Pothole-seg'
os.makedirs(drive_folder, exist_ok=True)

!cp -r "{calib_path}" "{drive_folder}/pothole_calib_images"


### **4. Train YOLOv8 Model**

We'll use YOLOv8s (small) for a good balance of speed and accuracy. Adjust epochs and imgsz as needed.

In [ ]:
from ultralytics import YOLO
import os

# === CORRECTED FOR SEGMENTATION ===
# Load a pretrained SEGMENTATION model (not detection)
model = YOLO('yolov8s-seg.pt')  # Note the "-seg" suffix

# Train the model
results = model.train(
    data=f"{filtered_path}/data.yaml",   # path to data.yaml
    epochs=60,                               # 60 epochs is good for segmentation
    imgsz=640,                               # input image size
    batch=16,                                # adjust based on GPU memory
    name='pothole_segmentation',             # experiment name
    device=0,                                # use GPU if available
    workers=8,                               # parallel workers
    patience=10,                             # early stopping patience
    save_period=5,                           # save checkpoint every 5 epochs
    seed=42,                                 # reproducibility
    exist_ok=True,                           # overwrite existing run
)

### **5. Validate Model**

In [ ]:
#!yolo task=detect mode=val model=/content/runs/detect/pothole_detection/weights/best.pt data={dataset.location}/data.yaml
metrics = model.val()
print(f"mAP50-95 (Box): {metrics.box.map:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

### **6. Test on Sample Images**

Run inference on a few validation images and display results.

In [ ]:
!yolo predict task=segment \
    model=runs/segment/pothole_segmentation/weights/best.pt \
    source={filtered_path}/test/images \
    save= True \
    name=pothole_seg_test

In [ ]:
# Show first 10 results
import glob
from IPython.display import Image, display

pred_dir = 'runs/segment/pothole_seg_test'
for img_file in sorted(glob.glob(f'{pred_dir}/*.jpg'))[:10]:
    display(Image(filename=img_file, height=400))
    print("\n")

### **7. Export Model to ONNX (for Hailo Compilation)**

In [ ]:

best_pt = 'runs/segment/pothole_segmentation/weights/best.pt'  # FIXED
model = YOLO(best_pt)
model.export(format='onnx', imgsz=640, opset=11, simplify=True)

### Save everything to **Google Drive**

In [ ]:
# --- 8. Save everything to Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/Pothole-seg'
os.makedirs(drive_folder, exist_ok=True)

# 8a. Copy best weights and ONNX
!cp runs/segment/pothole_segmentation/weights/best.pt "{drive_folder}/best.pt"
!cp runs/segment/pothole_segmentation/weights/best.onnx "{drive_folder}/best.onnx"

# 8b. Copy all training plots and metrics
!cp -r runs/segment/pothole_segmentation/*.png "{drive_folder}/"
!cp -r runs/segment/pothole_segmentation/*.csv "{drive_folder}/" 2>/dev/null

# 8c. Copy predicted images
pred_src = 'runs/segment/pothole_seg_test'
if os.path.exists(pred_src):
    !cp -r "{pred_src}" "{drive_folder}/predicted_test_images"

# 8d. (Optional) zip the entire runs folder for complete backup
!zip -r "{drive_folder}/pothole_seg_runs.zip" runs/segment/pothole_segmentation/ runs/segment/pothole_seg_test/

# 8e. Also save the filtered dataset (the one we actually trained on)
!cp -r "{filtered_path}" "{drive_folder}/filtered_dataset"

print(f"All files saved to {drive_folder}")


The ONNX file will be saved as `best.onnx` in the same folder as `best.pt`.

### **8. Download ONNX Model**

### **Visualize Training Results**

During training, Ultralytics saves various plots to the experiment directory (e.g., `runs/detect/pothole_segmentation`). Let's display some of the most important ones to review the training performance.

In [ ]:
import glob
from IPython.display import Image, display

results_dir = 'runs/segment/pothole_segmentation/'  # FIXED
print("Displaying key training result plots:")

for plot_name, title in [('results.png', 'Overall Training Metrics'),
                         ('MaskF1_curve.png', 'F1-Score Curve'),
                         ('confusion_matrix.png', 'Confusion Matrix'),
                         ('MaskP_curve.png', 'Precision-Confidence Curve'),
                         ('MaskR_curve.png', 'Recall-Confidence Curve')]:
    img_paths = glob.glob(f'{results_dir}/{plot_name}')
    if img_paths:
        print(f"\n--- {title} ---")
        display(Image(filename=img_paths[0], width=800))
    else:
        print(f"{plot_name} not found.")


### **10. Next Steps: Hailo Conversion**

On an x86 Ubuntu machine (or in another Colab if you install Hailo tools), you'll compile the ONNX model to a HEF file using the Hailo Dataflow Compiler. For example:
hailo compiler --hw-arch hailo8l --ckpt best.onnx --output-hef-path pothole_model.hef

Then deploy the .hef on your Raspberry Pi 5 with the Hailo‑8 accelerator using the HailoRT API or the hailo-rpi5-examples scripts.